# Task 4: Logistic Regression for Sentiment Classification

## Objective

We build a simple sentiment classifier.

We use:

* TF-IDF for text features
* Logistic Regression for classification

We predict:

* 1 = Positive
* 0 = Negative

---

# Step 1: Import Libraries

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

---

# Step 2: Load Dataset

In [2]:
DATA_DIR = Path.cwd() / "datasets"

df = pd.read_csv(DATA_DIR / "imdb_balanced_10k.csv")

In [3]:
print("Total reviews:", len(df))

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nFirst review preview:")
print(df["text"].iloc[0][:300])

print("\nFirst label:", df["label"].iloc[0])

Total reviews: 10000

Label distribution:
label
0    5000
1    5000
Name: count, dtype: int64

First review preview:
Unreal "movie", what were these people on?? A mix of French Upstairs Downstairs, mating horses,porn (not suggested, its pretty full on for a film) & bestiality with a bit of Benny Hill music & chase scenes thrown in, its sounds crazy & its even more so to watch. **spoiler** It plods along in a tedio

First label: 0


---

# Step 3: Split Train and Test Data

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [5]:
print("Testing samples:", len(X_test))

print("\nFirst training review:")
print(X_train.iloc[0][:300])

Testing samples: 2000

First training review:
Not only did they get the characters all wrong, not only do the voices suck, not only do the writers seriously need to get girlfriends, not only are the drawings really crude, but it seems like it was mainly created for ages 1-6. The only episode I've ever seen of this show that kept me watching, wa


---

# Step 4: Convert Text into TF-IDF Features

In [6]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

In [7]:
print("Training shape:", X_train_tfidf.shape)

print("Testing shape:", X_test_tfidf.shape)

print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

Training shape: (8000, 5000)
Testing shape: (2000, 5000)
Vocabulary size: 5000


---

# Step 5: Inspect Vocabulary

In [8]:
feature_names = vectorizer.get_feature_names_out()

print("First 20 vocabulary words:")
print(feature_names[:20])

First 20 vocabulary words:
['00' '000' '10' '100' '11' '12' '13' '14' '15' '16' '17' '18' '1930s'
 '1940' '1945' '1950' '1950s' '1953' '1959' '1960s']


---

# Step 6: Train Logistic Regression

In [9]:
model = LogisticRegression()

model.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


---

# Step 7: Make Predictions

In [10]:
y_pred = model.predict(X_test_tfidf)

In [11]:
print("First 20 predictions:")
print(y_pred[:20])

print("\nFirst 20 true labels:")
print(y_test.iloc[:20].values)

First 20 predictions:
[0 1 0 1 0 1 0 1 1 1 0 1 1 1 1 0 1 1 0 0]

First 20 true labels:
[0 1 0 1 0 0 0 1 0 1 0 1 1 1 1 1 1 1 0 0]


---

# Step 8: Evaluate Accuracy

In [12]:
accuracy = accuracy_score(y_test, y_pred)

In [13]:
print("Classification Accuracy:", round(accuracy, 4))

Classification Accuracy: 0.8885


---

# Step 9: Print Full Classification Report

In [14]:
print("Classification Report:\n")

print(classification_report(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.87      0.89      1000
           1       0.88      0.90      0.89      1000

    accuracy                           0.89      2000
   macro avg       0.89      0.89      0.89      2000
weighted avg       0.89      0.89      0.89      2000



---

# Step 10: Compare Real Reviews with Predictions

In [15]:
for i in range(5):
    print(f"\nReview {i+1}")
    print("-" * 60)

    print(X_test.iloc[i][:400])

    print("\nTrue Label:", y_test.iloc[i])

    print("Predicted Label:", y_pred[i])

    if y_test.iloc[i] == y_pred[i]:
        print("Result: Correct")
    else:
        print("Result: Incorrect")


Review 1
------------------------------------------------------------
At the name of Pinter, every knee shall bow - especially after his Nobel Literature Prize acceptance speech which did little more than regurgitate canned, by-the-numbers, sixth-form anti-Americanism. But this is even worse; not only is it a tour-de-force of talentlessness, a superb example of how to get away with coasting on your decades-old reputation, but it also represents the butchery of a sup

True Label: 0
Predicted Label: 0
Result: Correct

Review 2
------------------------------------------------------------
It's always difficult to put a stamp on any film as being 'the best,' whether of all time, a certain genre, or what have you, but I believe a strong argument could be made that in fact, Laputa is the greatest animated film ever made. It is in my mind the masterwork of Hayao Miyazaki, the most talented of Japan's animated directors, and it best captures his strengths as a director, storyteller, and

True 

---

# Step 11: Check Important Words

We inspect which words have strong positive or negative weights.

In [16]:
coefficients = model.coef_[0]

top_positive_idx = np.argsort(coefficients)[-10:]

top_negative_idx = np.argsort(coefficients)[:10]

In [17]:
print("Top Positive Words:")

for idx in reversed(top_positive_idx):
    print(feature_names[idx], round(coefficients[idx], 4))

Top Positive Words:
great 4.7839
excellent 3.9654
best 3.5855
wonderful 3.0232
perfect 2.9379
love 2.8739
favorite 2.8591
loved 2.7688
amazing 2.651
superb 2.5692


In [18]:
print("Top Negative Words:")

for idx in top_negative_idx:
    print(feature_names[idx], round(coefficients[idx], 4))

Top Negative Words:
worst -5.722
bad -5.4692
awful -3.8814
waste -3.7853
poor -3.5333
boring -3.5184
worse -3.3223
minutes -3.2507
terrible -3.204
unfortunately -2.9914


---

# Step 12: Predict New Reviews

In [19]:
sample_reviews = [
    "This movie was amazing and unforgettable",
    "This was a terrible boring film",
    "The movie was okay but too long"
]

In [20]:
sample_tfidf = vectorizer.transform(sample_reviews)

sample_preds = model.predict(sample_tfidf)

In [21]:
for review, pred in zip(sample_reviews, sample_preds):
    print("\nReview:")
    print(review)

    print(
        "Predicted Sentiment:",
        "Positive" if pred == 1 else "Negative"
    )


Review:
This movie was amazing and unforgettable
Predicted Sentiment: Positive

Review:
This was a terrible boring film
Predicted Sentiment: Negative

Review:
The movie was okay but too long
Predicted Sentiment: Negative
